In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/PROG74040-AI-Text-Detection"
)

DATA_DIR = PROJECT_DIR / "data" / "splits"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "train.parquet"
VAL_FILE = DATA_DIR / "validation.parquet"
TEST_FILE = DATA_DIR / "test.parquet"

print("Train:", TRAIN_FILE)
print("Validation:", VAL_FILE)
print("Test:", TEST_FILE)

Train: /content/drive/MyDrive/PROG74040-AI-Text-Detection/data/splits/train.parquet
Validation: /content/drive/MyDrive/PROG74040-AI-Text-Detection/data/splits/validation.parquet
Test: /content/drive/MyDrive/PROG74040-AI-Text-Detection/data/splits/test.parquet


In [3]:
print("Train exists:", TRAIN_FILE.exists())
print("Validation exists:", VAL_FILE.exists())
print("Test exists:", TEST_FILE.exists())

Train exists: True
Validation exists: True
Test exists: True


In [4]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [5]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00


# Imports

In [6]:
import pandas as pd
import numpy as np

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Load the Parquet files

In [7]:
train_df = pd.read_parquet(TRAIN_FILE)
val_df = pd.read_parquet(VAL_FILE)
test_df = pd.read_parquet(TEST_FILE)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (341052, 3)
Validation: (73083, 3)
Test: (73083, 3)


,text,generated,text_length
0,The Importance of the Electoral College in Pre...,1,4878
1,Voting is one of the hardest choices a person ...,0,3114
2,Many kids believe that they should not have to...,0,4487
3,The author supports his or her idea that study...,0,2296
4,Automobiles have been all anyone talks about s...,0,2400


# Keep only two required columns

In [8]:
train_df = train_df[["text", "generated"]].copy()
val_df = val_df[["text", "generated"]].copy()
test_df = test_df[["text", "generated"]].copy()

In [9]:
train_df = train_df.rename(columns={"generated": "label"})
val_df = val_df.rename(columns={"generated": "label"})
test_df = test_df.rename(columns={"generated": "label"})

In [10]:
train_df.head()

,text,label
0,The Importance of the Electoral College in Pre...,1
1,Voting is one of the hardest choices a person ...,0
2,Many kids believe that they should not have to...,0
3,The author supports his or her idea that study...,0
4,Automobiles have been all anyone talks about s...,0


In [11]:
small_train_df = train_df.sample(
    n=10000,
    random_state=42
)

small_val_df = val_df.sample(
    n=2000,
    random_state=42
)

print("Small train:", small_train_df.shape)
print("Small validation:", small_val_df.shape)

Small train: (10000, 2)
Small validation: (2000, 2)


# Convert pandas to Hugging Face Dataset

In [12]:
small_train_ds = Dataset.from_pandas(
    small_train_df,
    preserve_index=False
)

small_val_ds = Dataset.from_pandas(
    small_val_df,
    preserve_index=False
)

small_train_ds

Dataset({
    features: ['text', 'label'],
    num_rows: 10000
})

# Load RoBERTa tokenizer

In [13]:
MODEL_NAME = "FacebookAI/roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [14]:
example = tokenizer(
    "This is a test sentence.",
    truncation=True,
    padding="max_length",
    max_length=256
)

print(example.keys())

KeysView({'input_ids': [0, 713, 16, 10, 1296, 3645, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

# Tokenization function

In [15]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

# Tokenize datasets

In [16]:
tokenized_train = small_train_ds.map(
    tokenize_function,
    batched=True
)

tokenized_val = small_val_ds.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [17]:
tokenized_train

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 10000
})

# Load RoBERTa model

In [18]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

# Define evaluation metrics

In [19]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    probabilities = torch.softmax(
        torch.tensor(logits),
        dim=-1
    )[:, 1].numpy()

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "roc_auc": roc_auc_score(labels, probabilities)
    }

# Training settings

In [20]:
training_args = TrainingArguments(
    output_dir="/content/roberta_test",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=50,

    report_to="none",

    fp16=True
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

In [22]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [23]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.054461,0.113963,0.979000,0.950202,0.992968,0.971114,0.998752


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
val_results = trainer.evaluate()

val_results

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.054461,0.113963,1,0.979000,0.950202,0.992968,0.971114,0.998752


{'eval_loss': 0.11396323889493942,
 'eval_accuracy': 0.979,
 'eval_precision': 0.9502018842530283,
 'eval_recall': 0.9929676511954993,
 'eval_f1': 0.9711141678129298,
 'eval_roc_auc': 0.9987517444480452}

In [25]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_ds = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

test_ds = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['text', 'label'],
    num_rows: 341052
})
Dataset({
    features: ['text', 'label'],
    num_rows: 73083
})
Dataset({
    features: ['text', 'label'],
    num_rows: 73083
})


In [26]:
tokenized_train_full = train_ds.map(
    tokenize_function,
    batched=True
)

tokenized_val_full = val_ds.map(
    tokenize_function,
    batched=True
)

tokenized_test_full = test_ds.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/341052 [00:00<?, ? examples/s]

Map:   0%|          | 0/73083 [00:00<?, ? examples/s]

Map:   0%|          | 0/73083 [00:00<?, ? examples/s]

In [27]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [28]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/PROG74040-AI-Text-Detection/models/roberta_checkpoints",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=2,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=500,

    report_to="none",

    fp16=True,

    save_total_limit=2
)

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_full,
    eval_dataset=tokenized_val_full,
    compute_metrics=compute_metrics
)

In [30]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.007955,0.023824,0.996141,0.990493,0.999228,0.994842,0.999087
2,0.001297,0.004356,0.998974,0.997762,0.999486,0.998623,0.999999


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [31]:
val_results = trainer.evaluate(tokenized_val_full)
val_results

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.001297,0.004356,2,0.998974,0.997762,0.999486,0.998623,0.999999


{'eval_loss': 0.004356315825134516,
 'eval_accuracy': 0.9989737695496901,
 'eval_precision': 0.9977623711529291,
 'eval_recall': 0.9994855589035055,
 'eval_f1': 0.9986232216613126,
 'eval_roc_auc': 0.9999986036775053}

In [32]:
tokenized_test_full

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 73083
})

In [33]:
test_results = trainer.evaluate(
    tokenized_test_full,
    metric_key_prefix="test"
)

test_results

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.001297,0.004893,2,0.998987,0.997653,0.999633,0.998642,0.999951


{'test_loss': 0.004893096629530191,
 'test_accuracy': 0.9989874526223609,
 'test_precision': 0.9976528404298236,
 'test_recall': 0.9996325285709036,
 'test_f1': 0.9986417033773862,
 'test_roc_auc': 0.9999509393197001}

In [34]:
ROBERTA_MODEL_DIR = MODEL_DIR / "roberta_final"

trainer.save_model(str(ROBERTA_MODEL_DIR))
tokenizer.save_pretrained(str(ROBERTA_MODEL_DIR))

print("RoBERTa model saved to:")
print(ROBERTA_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa model saved to:
/content/drive/MyDrive/PROG74040-AI-Text-Detection/models/roberta_final


In [35]:
roberta_results_df = pd.DataFrame([{
    "model": "RoBERTa",
    "accuracy": test_results["test_accuracy"],
    "precision": test_results["test_precision"],
    "recall": test_results["test_recall"],
    "f1_score": test_results["test_f1"],
    "roc_auc": test_results["test_roc_auc"]
}])

roberta_results_path = OUTPUT_DIR / "roberta_results.csv"

roberta_results_df.to_csv(
    roberta_results_path,
    index=False
)

roberta_results_df

,model,accuracy,precision,recall,f1_score,roc_auc
0,RoBERTa,0.998987,0.997653,0.999633,0.998642,0.999951
